<a href="https://colab.research.google.com/github/nyp-sit/it3103-2025s1/blob/main/week11_RNN/text_classification_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Classification using RNN model
In this lab exercise, we will learn to build LSTM layer from scratch using TF. 

## Setup

In [1]:
import os
import shutil
import tensorflow as tf

from datetime import datetime
import tensorflow as tf

### Download the IMDb Dataset
You will use the [Large Movie Review Dataset](http://ai.stanford.edu/~amaas/data/sentiment/). You will train a sentiment classifier model on this dataset.

In [2]:
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"

dataset = tf.keras.utils.get_file("aclImdb_v1.tar.gz", url,
                                    untar=True, cache_dir='.',
                                    cache_subdir='')

In [3]:
dataset_dir = os.path.join(os.path.dirname(dataset), 'aclImdb')
# dataset_dir = os.path.join(os.path.dirname(dataset), 'aclImdb_v1_extracted/aclImdb')
os.listdir(dataset_dir)

['imdb.vocab', 'imdbEr.txt', 'README', 'test', 'train']

Take a look at the `train/` directory. It has `pos` and `neg` folders with movie reviews labelled as positive and negative respectively. You will use reviews from `pos` and `neg` folders to train a binary classification model.

In [4]:
train_dir = os.path.join(dataset_dir, 'train')
os.listdir(train_dir)
test_dir = os.path.join(dataset_dir, 'test')
os.listdir(test_dir)

['labeledBow.feat', 'neg', 'pos', 'urls_neg.txt', 'urls_pos.txt']

The `train` directory also has additional folders which should be removed before creating training dataset.

In [5]:
remove_dir = os.path.join(train_dir, 'unsup')
shutil.rmtree(remove_dir)

Next, create a `tf.data.Dataset` using `tf.keras.preprocessing.text_dataset_from_directory`. You can read more about this utility from the [api documentation](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text_dataset_from_directory). 

Use the `train` directory to create both train and validation datasets with a split of 20% for validation. Also note that here we use a smaller batch size of 128, as our model now is more complex, and will use up some significant memory, leaving little room for larger batch size.

In [6]:
batch_size = 128
seed = 123
train_ds = tf.keras.preprocessing.text_dataset_from_directory(
    train_dir, batch_size=batch_size, validation_split=0.2, 
    subset='training', seed=seed)
val_ds = tf.keras.preprocessing.text_dataset_from_directory(
    train_dir, batch_size=batch_size, validation_split=0.2, 
    subset='validation', seed=seed)

Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


Take a look at a few movie reviews and their labels `(1: positive, 0: negative)` from the train dataset.


In [7]:
for text_batch, label_batch in train_ds.take(1):
    for i in range(3):
        print(label_batch[i].numpy(), text_batch[i].numpy())

1 b"I have watched this movie well over 100-200 times, and I love it each and every time I watched it. Yes, it can be very corny but it is also very funny and enjoyable. The camp shown in the movie is a real camp that I actually attended for 7 years and is portrayed as camp really is, a great place to spend the summer. Everyone who has ever gone to camp, wanted to go to camp, or has sent a child to camp should see this movie because it'll bring back wonderful memories for you and for your kids."
1 b'This movie is SOOOO funny!!! The acting is WONDERFUL, the Ramones are sexy, the jokes are subtle, and the plot is just what every high schooler dreams of doing to his/her school. I absolutely loved the soundtrack as well as the carefully placed cynicism. If you like monty python, You will love this film. This movie is a tad bit "grease"esk (without all the annoying songs). The songs that are sung are likable; you might even find yourself singing these songs once the movie is through. This m

### Configure the dataset for performance

These are two important methods you should use when loading data to make sure that I/O does not become blocking.

`.cache()` keeps data in memory after it's loaded off disk. This will ensure the dataset does not become a bottleneck while training your model. If your dataset is too large to fit into memory, you can also use this method to create a performant on-disk cache, which is more efficient to read than many small files.

`.prefetch()` overlaps data preprocessing and model execution while training. 

You can learn more about both methods, as well as how to cache data to disk in the [data performance guide](https://www.tensorflow.org/guide/data_performance).

In [8]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## Text preprocessing

Next, define the dataset preprocessing steps required for your sentiment classification model. Initialize a TextVectorization layer with the desired parameters to vectorize movie reviews. 

TextVectorization layer is a text tokenizer which breaks up the text into words (it is similar to Keras Tokenizer but implemented as a layer). You can read more about TextVectorization layer [here](https://www.tensorflow.org/api_docs/python/tf/keras/layers/experimental/preprocessing/TextVectorization).


In [9]:
# Vocabulary size and number of words in a sequence.
VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 200
# Use the text vectorization layer to normalize, split, and map strings to 
# integers.
# Set maximum_sequence length as all samples are not of the same length.
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB_SIZE, 
    output_sequence_length=MAX_SEQUENCE_LENGTH
)

# Make a text-only dataset (no labels) and call adapt to build the vocabulary.
text_ds = train_ds.map(lambda x, y: x)
vectorize_layer.adapt(text_ds)

In [10]:
print(len(vectorize_layer.get_vocabulary()))

10000


## Create a classification model

In [14]:
import tensorflow as tf


class CustomLSTMLayer(tf.keras.layers.Layer):
    """
    First-principles LSTM that honours Embedding(mask_zero=True) and
    iterates with `for t in tf.range(T)` so Autograph converts it to a
    tf.while_loop in graph mode.

    Args
    ----
    units             : hidden / cell size
    return_sequences  : if True  → (B,T,U) ; else → (B,U)
    """

    def __init__(self, units, return_sequences=False, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.return_sequences = return_sequences

    # ---------------------------------------------------------------- #
    # Weights:  x_t @ W  +  h_{t-1} @ U  + b      → gates [i,f,o,g]
    # ---------------------------------------------------------------- #
    def build(self, input_shape):
        in_dim = int(input_shape[-1])

        self.W = self.add_weight(
            "W", shape=(in_dim, 4 * self.units),
            initializer="glorot_uniform"
        )
        self.U = self.add_weight(
            "U", shape=(self.units, 4 * self.units),
            initializer="orthogonal"
        )
        self.b = self.add_weight(
            "b", shape=(4 * self.units,),
            initializer="zeros"
        )
        super().build(input_shape)

    # ---------------------------------------------------------------- #
    # Forward
    # ---------------------------------------------------------------- #
    def call(self, inputs, mask=None, training=None):
        """
        inputs : (B, T, F)
        mask   : (B, T) – True on real tokens, False on padding
        """
        batch_size   = tf.shape(inputs)[0]
        time_steps   = tf.shape(inputs)[1]
        dtype        = inputs.dtype

        if mask is not None:
            # (B,T) ➜ (T,B,1) float32
            mask_f = tf.cast(tf.transpose(mask, [1, 0]), dtype)[:, :, None]
        else:
            mask_f = tf.ones((time_steps, batch_size, 1), dtype=dtype)

        h = tf.zeros((batch_size, self.units), dtype=dtype)
        c = tf.zeros((batch_size, self.units), dtype=dtype)

        ta = tf.TensorArray(dtype, size=time_steps)

        for t in tf.range(time_steps):
            x_t = tf.gather(inputs, t, axis=1)      # (B,F)
            m_t = tf.gather(mask_f, t, axis=0)      # (B,1)

            z   = tf.matmul(x_t, self.W) + tf.matmul(h, self.U) + self.b
            i, f, o, g = tf.split(z, 4, axis=1)
            i = tf.sigmoid(i) # Decides how much of the candidate content g should flow into the cell state.
            f = tf.sigmoid(f) # Decides what fraction of previous cell state cₜ₋₁ to keep. A value near 0 “forgets”; near 1 “remembers.”
            o = tf.sigmoid(o) # Controls how much of the new cell state passes to the hidden state hₜ (and thus outward to the rest of the network).
            g = tf.tanh(g)    # The fresh content that could be written to the cell state if the input gate allows it.

            c_new = f * c + i * g
            h_new = o * tf.tanh(c_new)

            # keep previous state on padded timesteps
            c = m_t * c_new + (1.0 - m_t) * c
            h = m_t * h_new + (1.0 - m_t) * h

            if self.return_sequences:
                ta = ta.write(t, h)

        if self.return_sequences:
            return tf.transpose(ta.stack(), [1, 0, 2])   # (B,T,U)
        return h                                         # (B,U)

    # ---------------------------------------------------------------- #
    # Keras bookkeeping
    # ---------------------------------------------------------------- #
    def compute_output_shape(self, input_shape):
        if self.return_sequences:
            return input_shape[:-1] + (self.units,)
        return (input_shape[0], self.units)

    def get_config(self):
        cfg = super().get_config()
        cfg.update(
            {"units": self.units, "return_sequences": self.return_sequences}
        )
        return cfg


In [15]:
# step 1: construct the model
EMBEDDING_DIM=128

model = tf.keras.Sequential(
    [
        vectorize_layer,
        tf.keras.layers.Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=EMBEDDING_DIM,
            mask_zero=True,
        ),

        # --------- custom RNNs replace the built-ins ------------
        CustomLSTMLayer(64, return_sequences=True),   # was tf.keras.layers.LSTM(64, return_sequences=True)
        CustomLSTMLayer(16),                          # was tf.keras.layers.LSTM(16)

        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)


## Compile and train the model

You will use [TensorBoard](https://www.tensorflow.org/tensorboard) to visualize metrics including loss and accuracy. Create a `tf.keras.callbacks.TensorBoard`.

In [16]:
root_logdir = os.path.join(os.curdir, "tb_logs")

def get_run_logdir():    # use a new directory for each run
    import time
    run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
    return os.path.join(root_logdir, run_id)

run_logdir = get_run_logdir()
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=run_logdir)
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath="bestcheckpoint.weights.h5",
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True)

Compile and train the model using the `Adam` optimizer and `BinaryCrossentropy` loss. 

In [17]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=False),
              metrics=['accuracy'])


In [18]:
model.fit(
    train_ds, 
    validation_data=val_ds,
    epochs=5, 
    callbacks=[tensorboard_callback, model_checkpoint_callback])

Epoch 1/5
157/157 [==============================] - 321s 2s/step - loss: 0.5544 - accuracy: 0.6965 - val_loss: 0.4326 - val_accuracy: 0.8208
Epoch 2/5
157/157 [==============================] - 318s 2s/step - loss: 0.3297 - accuracy: 0.8697 - val_loss: 0.3756 - val_accuracy: 0.8498
Epoch 3/5
157/157 [==============================] - 325s 2s/step - loss: 0.2683 - accuracy: 0.9009 - val_loss: 0.3999 - val_accuracy: 0.8318
Epoch 4/5
157/157 [==============================] - 323s 2s/step - loss: 0.2343 - accuracy: 0.9148 - val_loss: 0.4977 - val_accuracy: 0.7908
Epoch 5/5
157/157 [==============================] - 323s 2s/step - loss: 0.2377 - accuracy: 0.9107 - val_loss: 0.4297 - val_accuracy: 0.8452


Visualize the model metrics in TensorBoard.

In [19]:
%load_ext tensorboard
%tensorboard --logdir tb_logs

Reusing TensorBoard on port 6006 (pid 35528), started 6 days, 0:59:48 ago. (Use '!kill 35528' to kill it.)

The model reaches a validation accuracy of around 85% after 1 epoch of training.

Note: Your results may be a bit different, depending on how weights were randomly initialized before training the embedding layer. 


Let's evaluate the model on our test dataset.

In [20]:
test_ds = tf.keras.preprocessing.text_dataset_from_directory(
    test_dir, 
    batch_size=128)

Found 25000 files belonging to 2 classes.


In [21]:
test_loss, test_acc = model.evaluate(test_ds)

print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

196/196 [==============================] - 44s 216ms/step - loss: 0.4624 - accuracy: 0.8326
Test Loss: 0.46241363883018494
Test Accuracy: 0.8326399922370911


Here we show how we can use get all the individual predictions for the test_ds and use the predictions to plot the confusion_matrix and classification report to allow us to have better insight.

In [22]:
import numpy as np

y_preds = np.array([])
y_labels = np.array([])
count = 0
for texts, labels in test_ds:
    preds = model.predict(texts)
    preds = (preds >= 0.5).reshape(-1)
    y_preds = np.concatenate((y_preds, preds), axis=0)
    y_labels = np.concatenate((y_labels, labels), axis=0)

2/2 [==============================] - 0s 16ms/step


In [23]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_labels, y_preds))
print(confusion_matrix(y_true=y_labels, y_pred=y_preds))

              precision    recall  f1-score   support

         0.0       0.84      0.83      0.83     12500
         1.0       0.83      0.84      0.83     12500

    accuracy                           0.83     25000
   macro avg       0.83      0.83      0.83     25000
weighted avg       0.83      0.83      0.83     25000

[[10319  2181]
 [ 2003 10497]]



Let's go ahead and save our model. You will see that our model achieve an accuracy of around 82%. 

In [24]:
model.save('sentiment_model.keras')

Now let us put our model in use!!  We will first load our saved model.


In [25]:
loaded_model = tf.keras.models.load_model('sentiment_model.keras')

TypeError: Cannot deserialize object of type `CustomLSTMLayer`. If `CustomLSTMLayer` is a custom class, please register it using the `@keras.saving.register_keras_serializable()` decorator.

In [26]:
loaded_model.summary()

NameError: name 'loaded_model' is not defined

Run the following cell and type in your own text at the prompt:

In [ ]:
text = input("Write your review here:")

In [ ]:
pred = loaded_model.predict(tf.convert_to_tensor([text]))[0]
if pred >= 0.5: 
    print('positive sentiment')
else:
    print('negative sentiment')

1/1 [==============================] - 0s 21ms/step
positive sentiment


In [ ]:
model = tf.keras.Sequential([
    vectorize_layer,
    tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, mask_zero=True),
    tf.keras.layers.SimpleRNN(64,  return_sequences=True),
    tf.keras.layers.SimpleRNN(16),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
model = tf.keras.Sequential([
    vectorize_layer,
    tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, mask_zero=True),
    tf.keras.layers.LSTM(64,  return_sequences=True),
    tf.keras.layers.LSTM(16),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
model = tf.keras.Sequential([
    vectorize_layer,
    tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, mask_zero=True),
    tf.keras.layers.GRU(64,  return_sequences=True),
    tf.keras.layers.GRU(16),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])